# Optional strengthening experiments

**Paper:** *Where the Dimension Hides: The Approximation Constant of Kolmogorov–Arnold Networks*
**Author:** Haydar Kılıç · ORCID 0000-0002-2551-3772

None of these is required for submission. Each closes a gap the manuscript currently declares
as a limitation. They are ordered by value per hour.

| Part | Item | Closes | Cost |
|---|---|---|---|
| **D4** | Certificate for the two-layer family, four schemes | §5.2's central open question; §5.6 limitation | ~1 GPU-h |
| **D1** | Duplicate-edge definition sweep | §4.10's unresolved contradiction | ~0.5 GPU-h |
| **D2** | Hidden width decoupled from dimension | §5.6's depth limitation | ~1.5 GPU-h |
| **D3** | Seed subset re-run for hardware replication | §5.6's single-machine limitation | ~10 min |

---

### A correction to item D2 as it was written

The checklist called for "a width probe with trainable, independently useful outputs". That
experiment cannot exist. For any depth-one KAN layer followed by a linear read-out,

$$\sum_{o} w_o \sum_p \phi_{po}(x_p) \;=\; \sum_p \Big[\sum_o w_o\,\phi_{po}(x_p)\Big],$$

which is again a sum of univariate functions. Making the read-out trainable changes nothing:
**at depth one the hypothesis class is additive no matter how wide the layer is.** That is
precisely why `WidthKAN` could only show that redundant edges are harmless.

The question the limitation actually needs answering is different, and is answerable: *does a
wider inner layer hurt, at fixed depth two?* D2 below sweeps the hidden width $H$ of `KANet`
**independently of the dimension**, instead of pinning it at $2\dime+1$. That separates width
from dimension inside the two-layer architecture, which is where the unexplained slope lives.

### The idea behind D4

Our alternating solve failed because it started from a random inner map and was beaten by
AdamW in 84% of cells. One of the four schemes below removes that failure mode by
construction: **warm-start the alternating scheme from the trained AdamW solution.** The exact
outer solve can then only reduce the training error, so the result is guaranteed to be at least
as good as AdamW, and any further decrease is a certified optimisation deficit.

In [ ]:
import os, json, math, time, random, itertools
import numpy as np
import pandas as pd
from scipy import stats
import torch, torch.nn as nn, torch.nn.functional as F

SCALE = "full"          # "smoke" (CPU, minutes) | "full" (GPU, ~3 h)
REQUIRE_PERSISTENT = True

def _resolve_out():
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        d = "/content/drive/MyDrive/KAN_JMLR/optional_outputs"
        os.makedirs(d, exist_ok=True)
        p = os.path.join(d, ".w"); open(p, "w").write("ok"); os.remove(p)
        print(f"[storage] persistent: {d}"); return d
    except ImportError:
        d = os.path.abspath("./optional_outputs")
        print(f"[storage] local: {d}"); return d
    except Exception as e:
        if REQUIRE_PERSISTENT:
            raise RuntimeError(
                f"Drive could not be mounted ({type(e).__name__}). Results would go to "
                "/content, which Colab deletes on recycle. Mount Drive, or set "
                "REQUIRE_PERSISTENT = False and download the CSVs before closing.")
        return os.path.abspath("/content/optional_outputs")

OUT = _resolve_out(); os.makedirs(OUT, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.use_deterministic_algorithms(True, warn_only=True)

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

SCALES = {
 "smoke": dict(dims=[2,4,8], hidden=[2,4,8], grids=[4,8], orders=[3], seeds=[0],
               epochs=40, n_base=2_000, real=["synthetic"], dup_edges=[0,4,16]),
 "full":  dict(dims=[2,4,8,16,32], hidden=[2,4,8,16,32,64], grids=[4,8,16], orders=[3],
               seeds=[0,1,2], epochs=500, n_base=16_000,
               real=["synthetic","airfoil","concrete","energy","wine","superconduct"],
               dup_edges=[0,2,4,8,16,32]),
}
S = SCALES[SCALE]
BATCH, LR, WD, CAP = 2_048, 3e-3, 1e-5, 0.25
print(f"scale={SCALE}  device={DEVICE}")

## 1. Primitives

In [ ]:
class KANLayer(nn.Module):
    def __init__(self, i, o, grid=8, order=3, gr=(-1., 1.)):
        super().__init__()
        self.order, self.grid, self.grid_range = order, grid, gr
        step = (gr[1] - gr[0]) / grid
        self.register_buffer("knots", torch.arange(-order, grid + order + 1) * step + gr[0])
        self.coeff = nn.Parameter(torch.randn(i, o, grid + order) * 0.1)
        self.w_res = nn.Parameter(torch.randn(i, o) * 0.1)
    def _basis(self, x):
        g0, g1 = self.grid_range; eps = 1e-6 * (g1 - g0)
        x = x.clamp(g0 + eps, g1 - eps).unsqueeze(-1); t = self.knots
        b = ((x >= t[:-1]) & (x < t[1:])).to(x.dtype)
        for k in range(1, self.order + 1):
            ld = (t[k:-1] - t[:-(k+1)]).clamp_min(1e-8)
            rd = (t[k+1:] - t[1:-k]).clamp_min(1e-8)
            b = (x - t[:-(k+1)]) / ld * b[..., :-1] + (t[k+1:] - x) / rd * b[..., 1:]
        return b
    def forward(self, x):
        return (torch.einsum("bik,iok->bio", self._basis(x), self.coeff)
                + F.silu(x).unsqueeze(-1) * self.w_res).sum(1)


class EdgeKAN(nn.Module):
    def __init__(self, d, G, k):
        super().__init__(); self.kan = KANLayer(d, 1, G, k)
    def forward(self, x): return self.kan(x)


class KANetH(nn.Module):
    """Two-layer probe with the hidden width H supplied explicitly.

    The study pinned H = 2d+1, which ties width to dimension. Sweeping H independently is
    what separates the two inside the two-layer architecture.  E = H(d+1).
    """
    def __init__(self, d, H, G, k):
        super().__init__()
        self.H, self.edges = H, H * (d + 1)
        self.l1 = KANLayer(d, H, G, k); self.l2 = KANLayer(H, 1, G, k)
    def forward(self, x): return self.l2(self.l1(x))


count_params = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)
relative_l2  = lambda p, y: (torch.linalg.norm(p - y) /
                             torch.linalg.norm(y).clamp_min(1e-12)).item()

def _additive(X):
    psi = lambda t: torch.sin(math.pi * t) + 0.3 * torch.cos(3 * math.pi * t)
    return psi(X).sum(1, keepdim=True) / math.sqrt(X.shape[1])

def _ka_struct(X, seed):
    d = X.shape[1]; H = 2 * d + 1
    g = torch.Generator().manual_seed(seed + 31)
    a = torch.randn(H, d, generator=g); ph = torch.rand(H, d, generator=g) * 2 * math.pi
    c = torch.randn(H, 1, generator=g) / math.sqrt(H)
    inner = (a.unsqueeze(0) * torch.sin(math.pi * X.unsqueeze(1) + ph.unsqueeze(0))).sum(-1)
    return torch.tanh(inner / math.sqrt(d)) @ c

def make_synth(kind, d, n_tr, n_te, seed):
    g = torch.Generator().manual_seed(seed + 1000)
    Xtr = torch.rand(n_tr, d, generator=g) * 2 - 1
    Xte = torch.rand(n_te, d, generator=g) * 2 - 1
    fn = _additive if kind == "additive" else (lambda X: _ka_struct(X, seed))
    ytr, yte = fn(Xtr), fn(Xte)
    mu, sd = ytr.mean(), ytr.std().clamp_min(1e-8)
    to = lambda t: t.to(DEVICE, torch.float32)
    return to(Xtr), to((ytr - mu) / sd), to(Xte), to((yte - mu) / sd)

def train_eval(model, data, epochs, seed, lr=LR, batch=BATCH, return_model=False):
    seed_everything(seed)
    Xtr, ytr, Xte, yte = data
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WD)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    n = Xtr.shape[0]; g = torch.Generator(device="cpu").manual_seed(seed)
    model.train(); last = float("nan")
    for _ in range(epochs):
        perm = torch.randperm(n, generator=g).to(Xtr.device); tot = nb = 0
        for i in range(0, n, batch):
            idx = perm[i:i+batch]
            opt.zero_grad(set_to_none=True)
            loss = F.mse_loss(model(Xtr[idx]), ytr[idx]); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        sch.step(); last = tot / max(nb, 1)
    model.eval()
    with torch.no_grad():
        ch = lambda X: torch.cat([model(X[i:i+8192]) for i in range(0, X.shape[0], 8192)])
        r = dict(rel_l2=relative_l2(ch(Xte), yte), rel_l2_train=relative_l2(ch(Xtr), ytr),
                 train_mse=last)
    return (r, model) if return_model else r

print("primitives ready")

## 2. D4 — a working certificate for the two-layer family

Four schemes, each producing a training error that is *attainable* by the architecture. Scheme
(d) is the one with a guarantee: starting from the AdamW solution, an exact outer solve can only
reduce the training error, so the certificate is at least as good as AdamW by construction and
any gap is a certified optimisation deficit.

In [ ]:
def outer_solve(model, Xtr, ytr, ridge=1e-8, rank=None, dtype=torch.float64):
    """Exact outer-layer solve with the inner map frozen.

    ridge : Tikhonov term, scaled by the mean diagonal of the Gram matrix.
    rank  : if given, truncate the SVD to this many components instead of ridging.
    """
    with torch.no_grad():
        Z = model.l1(Xtr)
        B = model.l2._basis(Z).reshape(Z.shape[0], -1)
        Phi = torch.cat([B, F.silu(Z)], 1).to(dtype)
        y = ytr.to(dtype)
        if rank is not None:
            U, Sv, Vh = torch.linalg.svd(Phi, full_matrices=False)
            r = min(rank, Sv.shape[0])
            sol = Vh[:r].T @ ((U[:, :r].T @ y) / Sv[:r].unsqueeze(-1))
        else:
            G = Phi.T @ Phi
            G = G + ridge * torch.eye(G.shape[0], dtype=dtype, device=G.device) * G.diagonal().mean()
            sol = torch.linalg.solve(G, Phi.T @ y)
        nb = model.l2.grid + model.l2.order; H = Z.shape[1]
        model.l2.coeff.copy_(sol[:H*nb].reshape(H, 1, nb).float())
        model.l2.w_res.copy_(sol[H*nb:].reshape(H, 1).float())
        return relative_l2((Phi @ sol).float(), ytr)


def certify(scheme, d, H, G, k, data, seed, rounds=6, inner=100, base_model=None):
    """Return (train, test) relative L2 for one certification scheme."""
    Xtr, ytr, Xte, yte = data
    seed_everything(seed)
    model = KANetH(d, H, G, k).to(DEVICE) if base_model is None else base_model
    hist = []

    if scheme == "ridge_swept":
        best, best_r = float("inf"), None
        for r in (1e-10, 1e-8, 1e-6, 1e-4, 1e-2):
            m2 = KANetH(d, H, G, k).to(DEVICE); m2.load_state_dict(model.state_dict())
            e = outer_solve(m2, Xtr, ytr, ridge=r)
            if e < best: best, best_r, model = e, r, m2
        hist.append(best)
    elif scheme == "svd_truncated":
        best = float("inf")
        for frac in (0.25, 0.5, 0.75, 1.0):
            m2 = KANetH(d, H, G, k).to(DEVICE); m2.load_state_dict(model.state_dict())
            ncol = m2.l2._basis(m2.l1(Xtr[:8])).reshape(8, -1).shape[1] + H
            e = outer_solve(m2, Xtr, ytr, rank=max(1, int(frac * ncol)))
            if e < best: best, model = e, m2
        hist.append(best)
    elif scheme in ("alternating", "warm_start"):
        for _ in range(rounds):
            hist.append(outer_solve(model, Xtr, ytr, ridge=1e-6))
            for p in model.l2.parameters(): p.requires_grad_(False)
            opt = torch.optim.AdamW(model.l1.parameters(), lr=LR, weight_decay=WD)
            model.train()
            for _ in range(inner):
                opt.zero_grad(set_to_none=True)
                F.mse_loss(model(Xtr), ytr).backward(); opt.step()
            for p in model.l2.parameters(): p.requires_grad_(True)
        hist.append(outer_solve(model, Xtr, ytr, ridge=1e-6))

    model.eval()
    with torch.no_grad():
        te = relative_l2(model(Xte), yte)
    return min(hist), te, hist


def run_D4(S, out_csv):
    done = set(pd.read_csv(out_csv)["key"]) if os.path.exists(out_csv) else set()
    rows, t0 = [], time.time()
    dims = [d for d in S["dims"] if d >= 8] or S["dims"]
    for k, d, G, s in itertools.product(S["orders"], dims, S["grids"], S["seeds"][:1]):
        H = 2 * d + 1
        data = make_synth("kastruct", d, S["n_base"], S["n_base"] // 2, s)
        base_r, base_m = train_eval(KANetH(d, H, G, k), data, S["epochs"], s, return_model=True)
        for scheme in ("ridge_swept", "svd_truncated", "alternating", "warm_start"):
            key = f"D4|{scheme}|k{k}|d{d}|G{G}|s{s}"
            if key in done: continue
            bm = None
            if scheme == "warm_start":
                bm = KANetH(d, H, G, k).to(DEVICE); bm.load_state_dict(base_m.state_dict())
            tr, te, hist = certify(scheme, d, H, G, k, data, s, base_model=bm)
            rows.append(dict(key=key, scheme=scheme, k=k, d=d, H=H, grid=G, seed=s,
                             adamw_train=round(base_r["rel_l2_train"], 6),
                             adamw_test=round(base_r["rel_l2"], 6),
                             cert_train=round(tr, 6), cert_test=round(te, 6),
                             gap=round(base_r["rel_l2_train"] - tr, 6),
                             beats_adamw=bool(tr <= base_r["rel_l2_train"] + 1e-9),
                             monotone=bool(all(np.diff(hist) <= 1e-9)) if len(hist) > 1 else True))
        if rows:
            pd.DataFrame(rows).to_csv(out_csv, mode="a", index=False,
                                      header=not os.path.exists(out_csv)); rows = []
            print(f"  d={d} G={G} done, {(time.time()-t0)/60:.1f} min")
    print(f"D4 done in {(time.time()-t0)/60:.1f} min")


D4_CSV = os.path.join(OUT, f"D4_certificate_{SCALE}.csv")
run_D4(S, D4_CSV)

D4 = pd.read_csv(D4_CSV) if os.path.exists(D4_CSV) else pd.DataFrame()
if not D4.empty:
    print("\nD4 -- which scheme certifies? (beats_adamw must be True to certify anything)")
    print(D4.groupby("scheme").agg(cells=("gap", "size"),
                                   beats=("beats_adamw", "mean"),
                                   mean_gap=("gap", "mean"),
                                   monotone=("monotone", "mean")).round(4).to_string())

## 3. D1 — what does "duplicate edge" mean?

The manuscript reports that the duplicate arm of control C5 behaves opposite to the original
run. The disagreement is almost certainly a definition: "duplicate" can mean at least five
things, and they are not equivalent. This sweep runs all five on the same data.

In [ ]:
DATA_CACHE = os.path.join(OUT, "uci_cache"); os.makedirs(DATA_CACHE, exist_ok=True)
UCI_IDS = dict(airfoil=291, concrete=165, energy=242, wine=186, superconduct=464)

def load_data(name, seed=0, test_frac=0.3):
    if name == "synthetic":
        d, n = 6, 4_000
        g = torch.Generator().manual_seed(seed + 500)
        X = torch.rand(n, d, generator=g) * 2 - 1; y = _additive(X)
        ntr = int(n * (1 - test_frac))
        mu, sd = y[:ntr].mean(), y[:ntr].std().clamp_min(1e-8)
        to = lambda t: t.to(DEVICE, torch.float32)
        return to(X[:ntr]), to((y[:ntr]-mu)/sd), to(X[ntr:]), to((y[ntr:]-mu)/sd)
    path = os.path.join(DATA_CACHE, f"{name}.csv")
    if not os.path.exists(path):
        try:
            from ucimlrepo import fetch_ucirepo
        except ImportError:
            import subprocess, sys
            for extra in ([], ["--break-system-packages"]):
                if subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                                   "ucimlrepo"] + extra).returncode == 0: break
            from ucimlrepo import fetch_ucirepo
        ds = fetch_ucirepo(id=UCI_IDS[name])
        pd.concat([ds.data.features, ds.data.targets], axis=1).to_csv(path, index=False)
    df = pd.read_csv(path).select_dtypes(include=[np.number]).dropna()
    X = df.iloc[:, :-1].values.astype(np.float32); y = df.iloc[:, -1:].values.astype(np.float32)
    rng = np.random.default_rng(seed); idx = rng.permutation(len(X))
    ntr = int(len(X) * (1 - test_frac)); tr, te = idx[:ntr], idx[ntr:]
    mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-8
    ym, ys = y[tr].mean(), y[tr].std() + 1e-8
    to = lambda a: torch.tensor(a, dtype=torch.float32, device=DEVICE)
    sq = lambda a: np.tanh((a - mu) / sd / 3.0)
    return to(sq(X[tr])), to((y[tr]-ym)/ys), to(sq(X[te])), to((y[te]-ym)/ys)


def append_variant(Xtr, Xte, n_extra, variant, seed):
    """Five inequivalent definitions of an appended 'edge'."""
    if n_extra == 0: return Xtr, Xte
    g = torch.Generator(device="cpu").manual_seed(seed + 7)
    d = Xtr.shape[1]
    cols = torch.randint(0, d, (n_extra,), generator=g)
    def cat(X, extra): return torch.cat([X, extra.to(X.device)], 1)
    if variant == "noise":
        return (cat(Xtr, torch.rand(Xtr.shape[0], n_extra, generator=g) * 2 - 1),
                cat(Xte, torch.rand(Xte.shape[0], n_extra, generator=g) * 2 - 1))
    if variant == "dup_exact":                      # exact copy of an informative feature
        return cat(Xtr, Xtr[:, cols].cpu()), cat(Xte, Xte[:, cols].cpu())
    if variant == "dup_jitter":                     # copy plus small noise (as in the paper run)
        j = lambda X: X[:, cols].cpu() + 0.01 * (torch.rand(X.shape[0], n_extra,
                                                            generator=g) * 2 - 1)
        return cat(Xtr, j(Xtr)), cat(Xte, j(Xte))
    if variant == "dup_permuted":                   # copy with rows shuffled: destroys the signal
        pt = torch.randperm(Xtr.shape[0], generator=g); pe = torch.randperm(Xte.shape[0], generator=g)
        return cat(Xtr, Xtr[:, cols].cpu()[pt]), cat(Xte, Xte[:, cols].cpu()[pe])
    if variant == "dup_of_noise":                   # duplicate an appended NOISE column
        ntr = torch.rand(Xtr.shape[0], 1, generator=g) * 2 - 1
        nte = torch.rand(Xte.shape[0], 1, generator=g) * 2 - 1
        return cat(Xtr, ntr.repeat(1, n_extra)), cat(Xte, nte.repeat(1, n_extra))
    raise ValueError(variant)


def run_D1(S, out_csv):
    done = set(pd.read_csv(out_csv)["key"]) if os.path.exists(out_csv) else set()
    variants = ("noise", "dup_exact", "dup_jitter", "dup_permuted", "dup_of_noise")
    rows, t0 = [], time.time()
    for name in S["real"]:
        try: Xtr, ytr, Xte, yte = load_data(name)
        except Exception as e:
            print(f"  [{name}] unavailable ({type(e).__name__})"); continue
        for v, extra, s in itertools.product(variants, S["dup_edges"], S["seeds"]):
            key = f"D1|{name}|{v}|e{extra}|s{s}"
            if key in done: continue
            Xa, Xb = append_variant(Xtr, Xte, extra, v, s)
            seed_everything(s)
            r = train_eval(EdgeKAN(Xa.shape[1], 8, 3), (Xa, ytr, Xb, yte), S["epochs"], s)
            rows.append(dict(key=key, dataset=name, variant=v, extra=extra, seed=s,
                             d=Xa.shape[1], **r))
        if rows:
            pd.DataFrame(rows).to_csv(out_csv, mode="a", index=False,
                                      header=not os.path.exists(out_csv)); rows = []
            print(f"  {name} done, {(time.time()-t0)/60:.1f} min")
    print(f"D1 done in {(time.time()-t0)/60:.1f} min")


D1_CSV = os.path.join(OUT, f"D1_duplicate_variants_{SCALE}.csv")
run_D1(S, D1_CSV)

D1 = pd.read_csv(D1_CSV) if os.path.exists(D1_CSV) else pd.DataFrame()
if not D1.empty:
    rows = []
    for (ds, v), g in D1.groupby(["dataset", "variant"]):
        ag = g.groupby("extra").rel_l2.mean().reset_index().sort_values("extra")
        if len(ag) >= 4:
            rho, p = stats.spearmanr(ag.extra, ag.rel_l2)
            rows.append(dict(dataset=ds, variant=v, rho=round(float(rho), 3),
                             p=round(float(p), 4), positive=bool(rho > 0 and p < 0.1)))
    D1S = pd.DataFrame(rows)
    if D1S.empty:
        print("\nD1 -- no admissible cells (need at least four edge counts per dataset).")
    else:
        print("\nD1 -- Spearman trend by definition of an appended edge "
              "(law predicts rho > 0):")
        print(D1S.pivot_table(index="dataset", columns="variant",
                              values="rho").round(2).to_string())
        print("\n  fraction of datasets with the predicted positive trend, per definition:")
        print((D1S.groupby("variant").positive.mean() * 100).round(0).astype(int)
                .astype(str).add("%").to_string())
else:
    D1S = pd.DataFrame()

## 4. D2 — hidden width decoupled from dimension

`KANet` pins $H = 2\dime+1$, so width and dimension move together and the unexplained slope
could belong to either. Here $H$ is swept independently, at fixed depth two, under the
capacity-decoupled protocol. Two slopes are then separable: $\partial\log\relL/\partial\log H$
at fixed $\dime$, and $\partial\log\relL/\partial\log\dime$ at fixed $H$.

In [ ]:
def run_D2(S, out_csv):
    done = set(pd.read_csv(out_csv)["key"]) if os.path.exists(out_csv) else set()
    rows, t0 = [], time.time()
    jobs = sorted(itertools.product(S["orders"], S["dims"], S["hidden"],
                                    S["grids"], S["seeds"]),
                  key=lambda j: j[2] * (j[1] + 1) * (j[3] + j[0] + 1))
    for k, d, H, G, s in jobs:
        key = f"D2|k{k}|d{d}|H{H}|G{G}|s{s}"
        if key in done: continue
        E = H * (d + 1); params = E * (G + k + 1)
        n_tr = max(S["n_base"], math.ceil(params / CAP))
        data = make_synth("kastruct", d, n_tr, n_tr // 2, s)
        seed_everything(s)
        r = train_eval(KANetH(d, H, G, k), data, S["epochs"], s)
        rows.append(dict(key=key, k=k, d=d, H=H, grid=G, seed=s, edges=E, params=params,
                         n_train=n_tr, cap_ratio=round(params / n_tr, 4), **r))
        if len(rows) % 25 == 0:
            pd.DataFrame(rows).to_csv(out_csv, mode="a", index=False,
                                      header=not os.path.exists(out_csv)); rows = []
            print(f"  {(time.time()-t0)/60:.1f} min elapsed")
    if rows:
        pd.DataFrame(rows).to_csv(out_csv, mode="a", index=False,
                                  header=not os.path.exists(out_csv))
    print(f"D2 done in {(time.time()-t0)/60:.1f} min")


D2_CSV = os.path.join(OUT, f"D2_width_decoupled_{SCALE}.csv")
run_D2(S, D2_CSV)

D2 = pd.read_csv(D2_CSV) if os.path.exists(D2_CSV) else pd.DataFrame()
if not D2.empty:
    def slope(g, xcol, ycol="rel_l2"):
        gg = g.groupby(xcol)[ycol].mean().reset_index().sort_values(xcol)
        return (float(np.polyfit(np.log(gg[xcol]), np.log(gg[ycol]), 1)[0])
                if len(gg) >= 3 and (gg[ycol] > 0).all() else np.nan)
    print("\nD2 -- slope in H at fixed d (columns: grid):")
    print(D2.groupby(["d", "grid"]).apply(lambda g: round(slope(g, "H"), 2))
            .unstack().to_string())
    print("\nD2 -- slope in d at fixed H (columns: grid):")
    print(D2.groupby(["H", "grid"]).apply(lambda g: round(slope(g, "d"), 2))
            .unstack().to_string())
    print("\n  For reference, KANet with H tied to d gives about +0.9 on the test split.")

## 5. D3 — replication on this machine

Re-runs a small subset of the main study's cells and compares them against the values it
recorded. Run this on a **second machine** and compare the printed table with the one from the
first; agreement turns reproducibility into replication.

In [ ]:
def run_D3(S, out_csv, n_cells=12):
    rows = []
    for i, (k, d, G, s) in enumerate(itertools.product([3], [2, 8, 32], [8, 32], [0, 1])):
        if i >= n_cells: break
        data = make_synth("additive", d, S["n_base"], S["n_base"] // 2, s)
        seed_everything(s)
        r = train_eval(EdgeKAN(d, G, k), data, min(S["epochs"], 200), s)
        rows.append(dict(k=k, d=d, grid=G, seed=s, **{kk: round(v, 6) for kk, v in r.items()}))
    df = pd.DataFrame(rows); df.to_csv(out_csv, index=False)
    return df


D3_CSV = os.path.join(OUT, f"D3_replication_{SCALE}.csv")
D3 = run_D3(S, D3_CSV)
print("D3 -- replication subset (compare across machines):")
print(D3.to_string(index=False))
print(f"\ndevice: {DEVICE} | torch {torch.__version__} | "
      f"gpu: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'n/a'}")

## 6. Report

In [ ]:
L = [f"Optional strengthening run, scale = {SCALE}, device = {DEVICE}", "=" * 70, ""]

if not D4.empty:
    g = D4.groupby("scheme").agg(beats=("beats_adamw", "mean"), gap=("gap", "mean"))
    best = g.beats.idxmax()
    L += ["D4 -- certification of the two-layer family:"]
    for sch, r in g.iterrows():
        L.append(f"  {sch:15s} beats AdamW in {r.beats*100:3.0f}% of cells, "
                 f"mean train gap {r.gap:+.4f}")
    L += [f"  best scheme: {best}",
          ("  -> a working certificate exists; Section 5.2 can separate optimisation from "
           "approximation for KANet."
           if g.loc[best, "beats"] > 0.9 else
           "  -> still no scheme dominates AdamW; the limitation stands."), ""]

if not D1.empty and not D1S.empty:
    L += ["D1 -- definition of an appended edge:"]
    for v, r in D1S.groupby("variant").positive.mean().items():
        L.append(f"  {v:14s} predicted positive trend in {r*100:3.0f}% of datasets")
    L += ["  The definition that reproduces the original run is the one to adopt; if none",
          "  does, the duplicate arm should be dropped from the paper rather than reported.", ""]

if not D2.empty:
    L += ["D2 -- width decoupled from dimension:",
          "  compare the H-slope against the d-slope in the tables above. If the d-slope",
          "  persists at fixed H, the effect belongs to dimension; if it vanishes and the",
          "  H-slope carries it, the effect belongs to width and Section 5.2 can say so.", ""]

L += ["D3 -- replication subset written; run on a second machine and compare.", "",
      "Scope: all four are optional strengthenings, run after the pre-registered study.",
      "None revises the verdict.", "", "=" * 70,
      "DOWNLOAD BEFORE CLOSING", "=" * 70]
for f in sorted(os.listdir(OUT)):
    p = os.path.join(OUT, f)
    if os.path.isfile(p): L.append(f"  {os.path.getsize(p):>10,} B   {p}")

report = "\n".join(L); print(report)
open(os.path.join(OUT, f"optional_summary_{SCALE}.txt"), "w").write(report)

try:
    from google.colab import files as _f
    for _p in (D4_CSV, D1_CSV, D2_CSV, D3_CSV):
        if os.path.exists(_p): _f.download(_p)
except Exception as _e:
    print(f"(automatic download unavailable: {type(_e).__name__})")